# NB3: Does Dropout Still Help When There is More Data?

**COMP3620 Coursework 3: Dropout Detectives** (Kit Marrack, Holly Duncan, Isobel Martin, Owais Mahmood)

This notebook teaches our finding to a student who has read only our challenge statement. It takes about **30 minutes**. You will:

1. See what overfitting looks like, and how dropout works
2. **Predict** where dropout should help most, before seeing our results
3. Train a small network yourself, with and without dropout
4. Compare your prediction with our full results, and see how we explain them.

In [ ]:
# QUICK = True: loads our saved results and trains only a small example (under 5 minutes).
# QUICK = False: re-runs the full experiment (slow).
QUICK = True

# TODO: imports, device, and loading the shared code / cached results used by NB1 and NB2

## 1. Overfitting: memorising instead of learning (≈ 5 min)

[Explain overfitting in plain terms: a network that memorises its training images does well on them but badly on new ones. Show the gap between training and test accuracy.]

**Figure:** training and test accuracy per epoch for the no-dropout network on a small training set (from the NB1 cache). [Point out where test loss starts to rise while training accuracy keeps rising.]

In [ ]:
# TODO: load the NB1 cache and plot training vs test accuracy (and loss) per epoch, no dropout

## 2. What Dropout Does (≈ 5 min)

### The Idea:

Imagine revising for an exam where, every time you practise, someone covers up a random part of your notes. You can't rely on any one note being there, so you're forced to actually understand the topic instead of memorising specific answers.

Dropout does the same thing to a neural network. During training, it randomly switches off some of the hidden units, so the network can't rely on any single unit to memorise the training images. It has to spread what it learns across many units, which usually makes it better at classifying images it hasn't seen before.

### How it Works:

- During **training**, each hidden unit is switched off (its output is set to 0) with probability *p*, the **dropout rate**. With *p* = 0.5, about half the units are switched off.
- A **new random set** of units is switched off at **every training step**, so no unit can be relied on.
- During **testing**, **nothing is switched off**: the full network is used.

In PyTorch, `model.train()` turns dropout on and `model.eval()` turns it off. This is why our code calls `model.eval()` before measuring accuracy.

### See It for Yourself:

The cell below applies dropout with *p* = 0.5 to ten ones, twice in training mode and once in evaluation mode. Before you run it, think about which values you expect to see.

In [1]:
import torch, torch.nn as nn

torch.manual_seed(0)                            # so you see the same result every time you run it
drop = nn.Dropout(p=0.5)
x = torch.ones(10)

drop.train()                                    # training mode: dropout is on
print("Training mode, run 1:", drop(x))
print("Training mode, run 2:", drop(x))

drop.eval()                                     # evaluation mode: dropout is off
print("Evaluation mode:     ", drop(x))

Training mode, run 1: tensor([0., 0., 2., 0., 0., 0., 2., 2., 0., 2.])
Training mode, run 2: tensor([2., 2., 2., 0., 2., 2., 2., 2., 0., 0.])
Evaluation mode:      tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


Three Things to Notice:

1. In training mode, roughly half the values are **0**: those units have been switched off.
2. The two training runs switch off **different** units: a new random set is chosen every time.
3. In evaluation mode, **nothing changes**: every value is still 1.

But Why Are the Surviving Values **2**, not 1?

### Why the Surviving Values Are Scaled Up:

If half the units are switched off during training, the next layer only receives about half the signal it would get at test time, when every unit is on. The network would then see much bigger values during testing than it ever saw while training.

To avoid this, PyTorch multiplies the surviving units by $\frac{1}{1-p}$ during training. With *p* = 0.5, that is $\frac{1}{0.5} = 2$. On average, the output then stays the same as without dropout:

$$
\underbrace{(1-p)}_{\text{kept}} \times \frac{x}{1-p} \;+\; \underbrace{p}_{\text{dropped}} \times 0 \;=\; x
$$

So the network sees values of the same size in training and testing.

*Note:* UDL (§9.3.3) describes the same idea the other way round: it trains without scaling and instead multiplies the weights by (1 − *p*) at test time, which it calls the **weight scaling inference rule**. PyTorch scales during training instead, so that nothing needs to change at test time. The two give the same result.

**Quick Prediction:** In the next cell, dropout with *p* = 0.5 is applied to 10,000 ones. What fraction of the values do you think will be 0, and what will their **average** be?

In [2]:
x = torch.ones(10_000)
drop.train()
out = drop(x)

print(f"Fraction switched off: {(out == 0).float().mean():.3f}")
print(f"Average output:        {out.mean():.3f}")

Fraction switched off: 0.504
Average output:        0.991


About half the values are switched off, as expected for *p* = 0.5, but the average stays close to 1, because the surviving values were doubled.

### Where Dropout Sits in Our Network:

Our network has two hidden layers of 256 units. Dropout is applied after each hidden layer's ReLU, but **not** to the output layer: we always want all ten class scores.

    Linear(784 → 256) → ReLU → Dropout(p)
    Linear(256 → 256) → ReLU → Dropout(p)
    Linear(256 → 10)

With *p* = 0, dropout does nothing, which gives us our baseline network without dropout.

**Further Reading:** *Understanding Deep Learning*, §9.3.3; Srivastava et al. (2014), *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*, the original dropout paper.

## 3. Why the Amount of Data Might Matter (≈ 5 min)

Dropout helps a network that is overfitting. But does it help just as much when there is lots of training data? There are two reasons to expect that it should help **less** as the training set grows.

### Reason 1: Dropout Removes Kinks Between the Training Data:

Our network uses ReLU activations, so the function it learns is made of straight pieces joined at **kinks**. Near the training images, the loss pins the function down: the network has to get those images right. But between and far from the training images, nothing in the loss constrains it.

*Understanding Deep Learning* (§9.3.3, Figure 9.9) shows how unnecessary kinks can form there. Several hidden units can "conspire": one increases the slope, a second decreases it, and a third cancels the decrease, returning the function to its original path. Together they make a sharp local change that doesn't affect the training loss but is unlikely to generalise well.

Dropout breaks these conspiracies. If one of the units is switched off, the output changes a lot in the region where that unit was active, and the next gradient-descent step tries to correct this. Over time, the network stops relying on such combinations of units, and the unnecessary kinks between training points are gradually removed, even though they contribute nothing to the loss. Dropout also encourages smaller weights, so that switching any one unit on or off changes the output less.

**Our Reasoning:** UDL (§8.3.1) explains that more training data ensures the input space is **well sampled**. With more training images, there are fewer regions between and far from the data where unnecessary kinks can form, so the data itself constrains the function more, and there is less left for dropout to remove.

*Note:* UDL illustrates this with a one-dimensional example. Our images have 784 dimensions, so whether the same reasoning carries over is part of what our experiment tests.

### Reason 2: A Regulariser Matters Less When There is More Data:

Dropout is a **regulariser**: something added to training to stop the network fitting the training data too closely. A regulariser competes with the data. As a simple picture, think of the loss as

$$
L = \underbrace{\sum_{i=1}^{N} \ell_i}_{\text{fit the } N \text{ training images}} \;+\; \underbrace{\lambda \, R}_{\text{regulariser}}
$$

The data term grows with the number of training images *N*, while the regulariser does not. So the more data there is, the less the regulariser counts compared with the data.

Dropout does not add a term like this explicitly. However, Wager et al. (2013) show that for simpler models (generalised linear models, such as logistic regression), dropout is approximately equivalent to an L2 penalty that adapts to the data. Our network is deeper than this, so the result does not apply exactly, but it suggests the same reasoning should hold.

More data also means less overfitting to fix in the first place. UDL (§8.3.1) explains that with more training data, the fitted model depends less on the particular data it happened to see: its **variance** is reduced.

### What This Predicts:

If this reasoning is right, then as the training set grows:

1. **The benefit of dropout should shrink:** The improvement in test accuracy from using dropout should be largest for small training sets and close to zero for large ones.

2. **The best dropout rate should fall:** Less regularisation is needed, so the best *p* should get smaller.

3. **Dropout could even hurt:** If there is little overfitting left to fix, switching off units mainly makes the network harder to train, so test accuracy could be slightly lower with dropout than without.

These are the predictions we test in NB2.

**Further Reading:** *Understanding Deep Learning*, §9.3.3 and Figure 9.9 (dropout) and §8.3.1 (reducing variance); Wager, Wang and Liang (2013), *Dropout Training as Adaptive Regularization*, NeurIPS.

## 4. ✏️ Exercise 1: Make a Prediction (≈ 2 min)

Before you look at our results, use what you learnt in Sections 2 and 3 to make a prediction. Write your answers in the cell below; you will check them in Section 7.

1. We trained the same network with and without dropout on 500, 5,000 and 50,000 training images. For which of these sizes do you think dropout improved test accuracy **the most**? Why?

2. Do you think dropout could ever make test accuracy **worse**? If so, when?

**Your Predictions:**

1. Dropout helps most with ______ training images, because …
2. Dropout could make test accuracy worse when …

## 5. Try it yourself (≈ 5 min)

Train the same network on a **small** and a **large** training set, with and without dropout (one seed, a few epochs, so it runs quickly).

In [ ]:
# TODO: train 4 small runs (small/large training set × no dropout/dropout) with the shared training code
# and print a 2 × 2 table of test accuracies

## 6. Our full results (≈ 5 min)

**Figure:** dropout benefit (test accuracy with dropout minus without) against training-set size, mean ± standard deviation over [n] seeds, from the NB2 cache.

**Figure:** best dropout rate against training-set size.

[Describe the result in two or three sentences.]

In [ ]:
# TODO: load the NB2 cache and plot benefit vs training-set size (log x-axis, error bars), and best p vs size

## 7. Was your prediction right? (≈ 3 min)

[Compare with Exercise 1. Explain the result using the theory in Section 3, and say how we ruled out other explanations (e.g. that the benefit only shrinks because there is less overfitting left to fix).]

## 8. ✏️ Exercise 2

[Second short exercise, e.g. predict what happens to training and test accuracy with a very high dropout rate (p = 0.8) on a small training set, then check it in the NB1 results. Answer: training accuracy falls sharply and test accuracy falls too: the network underfits, so a smaller train–test gap is not always a sign of better generalisation.]

## 9. Summary: what this does and does not show (≈ 2 min)

- **What we found:** [one or two sentences]
- **What it does not show:** [scope, e.g. we tested fully connected networks on Fashion-MNIST only; we did not test CNNs or other datasets]
- **What next:** [one idea for a follow-up experiment]